# Pixel-wise runoff-onset vs climate anomaly correlations (exploratory)

Correlates the runoff-onset anomaly against each ERA5-Land variable's monthly anomaly per pixel across water years, on an
Equal-Earth grid derived on the fly from the ERA5-Land store (native 0.1°) and the public pyramid's ~10 km onset. The
correlations are computed one variable and water year at a time (~3 GB peak, tens of minutes of ERA5 reads) and kept as a
local zarr under `scratch/`; this notebook's own product, read by nothing else.

| | |
| --- | --- |
| Reads | the ERA5-Land icechunk repository of the version on Azure (acquisition and anomaly groups); the public multiscale pyramid, level 7 (anonymous); GMBA polygons from the web |
| Writes | `scratch/<version>_runoff_onset_and_era5_eqearth_anomaly_correlations.zarr` (local, gitignored) |
| Needs | the Azure SAS token |

In [ ]:
import cartopy.crs as ccrs
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import odc.geo.xr  # noqa: F401  -- registers the .odc accessor
import rioxarray  # noqa: F401
import xarray as xr
from global_snowmelt_runoff_onset.pyramid import open_pyramid_level

from gsro_analysis import era5, paths, settings

In [ ]:
config = settings.load_config()  # the dataset version lives in settings.CONFIG_FILE
correlations_path = paths.SCRATCH / f'{config.version}_runoff_onset_and_era5_eqearth_anomaly_correlations.zarr'
RECOMPUTE = False    # True recomputes the correlations even if the zarr exists
PYRAMID_LEVEL = 7    # ~10 km, the pyramid level closest to the ERA5-Land grid
print(correlations_path)

## 1. The ERA5-Land stack and the coarse onset on one Equal-Earth grid

In [ ]:
# the version's ERA5-Land acquisition: 8 monthly variables on (water_year, month, latitude, longitude), native 0.1 deg grid
era5_land_ds = era5.open_era5_land(config)
era5_land_ds

In [ ]:
# the target grid: one slab reprojected to Equal Earth (EPSG:8857) fixes the geobox every other slab is reprojected onto
climate_vars = [v for v in era5.VARIABLES if v in era5_land_ds]
water_years = [int(y) for y in era5_land_ds['water_year'].values]
months = list(era5_land_ds['month'].values)
template_ds = era5_land_ds[climate_vars[0]].isel(water_year=0, month=0).to_dataset().odc.reproject('EPSG:8857').compute()
geobox = template_ds.odc.geobox
print(geobox)
template_ds

In [ ]:
# the coarse onset from the public pyramid (anonymous), on the same grid; the onset anomaly is each year minus the
# per-pixel median over water years, kept only where the pixel has a valid median
coarse_onset_ds = open_pyramid_level(config, PYRAMID_LEVEL).rio.write_crs('EPSG:4326')[['runoff_onset', 'runoff_onset_median', 'temporal_resolution']]
onset_eqearth_ds = coarse_onset_ds.rio.reproject_match(template_ds).reindex(water_year=water_years)
onset_valid_da = onset_eqearth_ds['runoff_onset'].notnull()                            # (water_year, y, x)
onset_anomaly_da = (onset_eqearth_ds['runoff_onset'] - onset_eqearth_ds['runoff_onset'].median('water_year')).where(onset_eqearth_ds['runoff_onset_median'] > 0)
onset_anomaly_da

## 2. The correlations, streamed one variable and water year at a time

For every ERA5-Land variable and month: Pearson r across water years between the onset anomaly and the variable's anomaly
(the variable minus its per-pixel median over all water years), with the ERA5 field masked each year to the pixels that
have an onset value. One (variable, water year) slab is reprojected at a time; the whole stack as one lazy graph never
converged on a 16 GB machine.

In [ ]:
if RECOMPUTE or not correlations_path.exists():
    coords = {'water_year': water_years, 'y': template_ds['y'], 'x': template_ds['x']}
    ny, nx = geobox.shape
    correlations = {}
    for var in climate_vars:
        cube = np.empty((len(water_years), len(months), ny, nx), dtype='float32')
        for j, year in enumerate(water_years):
            slab_da = era5_land_ds[var].sel(water_year=year).load()               # (month, lat, lon), ~230 MB
            cube[j] = slab_da.odc.reproject(geobox).values
            del slab_da
        r = np.full((len(months), ny, nx), np.nan, dtype='float32')
        for mi in range(len(months)):
            climate_da = xr.DataArray(cube[:, mi], dims=('water_year', 'y', 'x'), coords=coords).where(onset_valid_da)
            climate_anomaly_da = climate_da - climate_da.median('water_year')
            r[mi] = xr.corr(onset_anomaly_da, climate_anomaly_da, dim='water_year').values
        correlations[var] = (('month', 'y', 'x'), r)
        del cube
        print(f'{var} done', flush=True)
    # the pyramid's yearly temporal resolution, correlated too and broadcast over month
    temporal_resolution_anomaly_da = onset_eqearth_ds['temporal_resolution'] - onset_eqearth_ds['temporal_resolution'].median('water_year')
    r_tres = xr.corr(onset_anomaly_da, temporal_resolution_anomaly_da, dim='water_year').values.astype('float32')
    correlations['temporal_resolution'] = (('month', 'y', 'x'), np.broadcast_to(r_tres, (len(months), ny, nx)).copy())

    correlations_ds = xr.Dataset(correlations, coords={'month': months, 'y': template_ds['y'], 'x': template_ds['x'], 'spatial_ref': template_ds['spatial_ref']})
    correlations_ds.attrs.update({
        'method': (f'Pearson r across water years of the runoff-onset anomaly (public pyramid level {PYRAMID_LEVEL}, nearest) vs each '
                   'ERA5-Land monthly anomaly, both on the Equal Earth grid of the reprojected ERA5 stack; ERA5 masked per year to '
                   'onset-valid pixels; anomalies vs the median over all water years; onset anomaly masked to valid-median pixels'),
        'water_years': water_years, 'dataset_version': config.version})
    for name in list(correlations_ds.variables):
        correlations_ds[name].encoding = {}
    correlations_ds.chunk({'month': -1, 'y': 512, 'x': 512}).to_zarr(correlations_path, mode='w')
    print(f'wrote {correlations_path}')

In [ ]:
correlations_ds = xr.open_zarr(correlations_path, decode_coords='all', chunks='auto').compute()
correlations_ds

In [ ]:
correlations_ds['temperature_2m'].plot.imshow(col='month', col_wrap=3, cmap='RdBu')

## 3. Regions and single ranges

With the RdBu colormap, red means an increase in the variable goes with an earlier runoff onset, blue with a later one.

In [ ]:
BBOXES = {'western_north_america': [-130, 30, -60, 75], 'high_mountain_asia': [65, 25, 110, 45], 'northern_europe_and_asia': [-10, 45, 80, 75]}
region_correlations_ds = correlations_ds.rio.clip_box(*BBOXES['high_mountain_asia'], crs='EPSG:4326')
region_correlations_ds

In [ ]:
for var in region_correlations_ds.data_vars:
    fig = region_correlations_ds[var].plot.imshow(col='month', col_wrap=3, cmap='RdBu', sharex=True, sharey=True,
                                                  subplot_kws={'projection': ccrs.EqualEarth()}, figsize=(12, 8))
    for ax in fig.axs.flatten():
        ax.set_aspect('equal')
        ax.set_title(ax.get_title().replace('month = ', ''))
        ax.gridlines(draw_labels=False)
    fig.fig.suptitle(f'Correlation between the {len(water_years)}-year runoff onset anomaly and\nthe ERA5-Land {var} anomaly', y=1.00)

In [ ]:
# GMBA Inventory v2.0 standard 300, read straight from EarthEnv
gmba_gdf = gpd.read_file('zip+' + settings.GMBA_URL)
gmba_gdf

In [ ]:
mountain_range_name = 'Brooks Range'
mountain_range_gdf = gmba_gdf[gmba_gdf['MapName'] == mountain_range_name]
mountain_range_correlations_ds = correlations_ds.rio.clip(mountain_range_gdf.to_crs(correlations_ds.rio.crs).geometry)
mountain_range_correlations_ds

In [ ]:
for var in mountain_range_correlations_ds.data_vars:
    fig = mountain_range_correlations_ds[var].plot.imshow(col='month', col_wrap=6, vmin=-1, vmax=1, cmap='RdBu_r', figsize=(12, 5))
    mean_correlation_da = mountain_range_correlations_ds[var].mean(dim=['x', 'y'])
    for i, ax in enumerate(fig.axs.flatten()[:mountain_range_correlations_ds.sizes['month']]):   # col_wrap=6 leaves 3 empty axes after 9 months
        mountain_range_gdf.to_crs(mountain_range_correlations_ds.rio.crs).boundary.plot(ax=ax, color='black', linewidth=1)
        month = ax.get_title().split('=')[-1]
        ax.set_title(f'{month}\navg_corr={mean_correlation_da.values[i]:.2f}')
        ax.axis('off')
    fig.fig.suptitle(var, y=1.02)

### One range's mean anomalies, month by month

The range-mean ERA5-Land anomaly (the anomaly group, clipped to the polygon on the native grid) against the range-mean
onset anomaly (the coarse pyramid), and their correlation per month and over the spring months.

In [ ]:
era5_anomaly_ds = era5.open_anomaly(config)                       # the anomaly group of the ERA5-Land icechunk repository
range_climate_anomaly_ds = (era5_anomaly_ds.rio.set_spatial_dims(x_dim='longitude', y_dim='latitude').rio.write_crs('EPSG:4326')
                            .rio.clip(mountain_range_gdf.geometry).mean(['latitude', 'longitude']).compute())
range_onset_anomaly_da = onset_anomaly_da.rio.clip(mountain_range_gdf.to_crs(onset_anomaly_da.rio.crs).geometry).mean(['x', 'y']).compute()
range_climate_anomaly_ds

In [ ]:
f, ax = plt.subplots(figsize=(5, 4))
ax.scatter(range_climate_anomaly_ds['temperature_2m'].sel(month='spring_month_1'), range_onset_anomaly_da)
ax.set_xlabel('spring month 1 temperature anomaly [K]')
ax.set_ylabel('runoff onset anomaly [days]')
ax.set_title(mountain_range_name)

In [ ]:
for month in months:
    r = xr.corr(range_climate_anomaly_ds['temperature_2m'].sel(month=month), range_onset_anomaly_da).values
    print(f'correlation for {month}: {r:.2f}')
r_spring = xr.corr(range_climate_anomaly_ds['temperature_2m'].sel(month=['spring_month_1', 'spring_month_2', 'spring_month_3']).mean('month'), range_onset_anomaly_da).values
print(f'correlation for the spring months: {r_spring:.2f}')